# 03 - Ensemble Methods & Threshold Calibration

This notebook analyzes the **CrossEncoder Reranker** scores to empirically determine
the optimal threshold for the **Answer Routing** mechanism in our RAG pipeline.

When the reranker score falls below this threshold, the system bypasses the retrieved
documents and lets the fine-tuned Qwen 1.7B answer from its own knowledge instead.

**Pipeline**: Hybrid Retrieval (FAISS + KG via RRF) → CrossEncoder Reranking → Threshold Gate → Generation

## 1. Setup & Data Loading

In [ ]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
import sys, os

# Add project root to path
sys.path.append(os.path.dirname(os.getcwd()))

from src.config import get_config
from src.rag.retriever import AdvancedRetriever

config = get_config()
retriever = AdvancedRetriever(config)

sns.set_theme(style='whitegrid', palette='muted')
random.seed(42)

In [ ]:
with open(config.paths.data_dir / 'qa_pairs.json', 'r', encoding='utf-8') as f:
    qa_pairs = json.load(f)

# Split into answerable vs unanswerable based on ground truth
UNANSWERABLE_LABEL = 'Il testo non fornisce informazioni utili.'
answerable = [item for item in qa_pairs if item['answer'] != UNANSWERABLE_LABEL]
unanswerable = [item for item in qa_pairs if item['answer'] == UNANSWERABLE_LABEL]

print(f'Total QA pairs: {len(qa_pairs)}')
print(f'Answerable:     {len(answerable)}')
print(f'Unanswerable:   {len(unanswerable)}')

# Sample (use all unanswerable since they are few)
sample_ans = random.sample(answerable, min(50, len(answerable)))
sample_unans = unanswerable  # use all

## 2. Score Collection

For each question, run the full retrieval pipeline and record the **top-1 CrossEncoder rerank score**.
- `y_true = 1` → answerable question (documents contain the answer)
- `y_true = 0` → unanswerable question (documents do NOT contain the answer)

In [ ]:
y_true = []
y_scores = []

print('Evaluating Answerable queries...')
for i, item in enumerate(sample_ans):
    chunks = retriever.retrieve(item['question'])
    if chunks:
        y_true.append(1)
        y_scores.append(chunks[0].get('rerank_score', -10.0))
    if (i + 1) % 10 == 0:
        print(f'  Progress: {i+1}/{len(sample_ans)}')

print(f'\nEvaluating Unanswerable queries...')
for i, item in enumerate(sample_unans):
    chunks = retriever.retrieve(item['question'])
    if chunks:
        y_true.append(0)
        y_scores.append(chunks[0].get('rerank_score', -10.0))

print(f'\nCollected {len(y_scores)} scores ({sum(y_true)} answerable, {len(y_true) - sum(y_true)} unanswerable)')

## 3. Score Distribution Analysis

Visualize the CrossEncoder score distributions for answerable vs. unanswerable queries.
A clear separation between the two distributions indicates that the reranker can discriminate well.

In [ ]:
scores_ans = [s for s, y in zip(y_scores, y_true) if y == 1]
scores_unans = [s for s, y in zip(y_scores, y_true) if y == 0]

print(f'Answerable   - Mean: {np.mean(scores_ans):.2f}, Median: {np.median(scores_ans):.2f}, Min: {np.min(scores_ans):.2f}, Max: {np.max(scores_ans):.2f}')
print(f'Unanswerable - Mean: {np.mean(scores_unans):.2f}, Median: {np.median(scores_unans):.2f}, Min: {np.min(scores_unans):.2f}, Max: {np.max(scores_unans):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# KDE Plot
ax = axes[0]
sns.kdeplot(scores_ans, label='Answerable', fill=True, alpha=0.4, color='#2ecc71', ax=ax)
sns.kdeplot(scores_unans, label='Unanswerable', fill=True, alpha=0.4, color='#e74c3c', ax=ax)
ax.axvline(x=0.5, color='navy', linestyle='--', lw=2, label='Chosen Threshold (0.5)')
ax.set_title('CrossEncoder Score Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Rerank Score')
ax.set_ylabel('Density')
ax.legend()

# Box Plot
ax = axes[1]
data = [scores_ans, scores_unans]
bp = ax.boxplot(data, labels=['Answerable', 'Unanswerable'], patch_artist=True,
                boxprops=dict(facecolor='lightblue'), medianprops=dict(color='red', lw=2))
ax.axhline(y=0.5, color='navy', linestyle='--', lw=2, label='Chosen Threshold (0.5)')
ax.set_title('Score Distribution (Box Plot)', fontsize=14, fontweight='bold')
ax.set_ylabel('Rerank Score')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(os.getcwd()), 'notebooks', 'plots', 'score_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. ROC Curve & Optimal Threshold (Youden's J)

The ROC curve plots the True Positive Rate vs. False Positive Rate at every possible threshold.
**Youden's J statistic** (`J = TPR - FPR`) identifies the point that maximizes the separation.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f'AUC:                {roc_auc:.4f}')
print(f'Youden\'s Optimal:   {optimal_threshold:.4f}')
print(f'Chosen Threshold:   0.5 (conservative, based on empirical observation)')

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='#e67e22', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.2f})')
ax.plot([0, 1], [0, 1], color='grey', lw=1.5, linestyle='--', label='Random Classifier')
ax.scatter([fpr[optimal_idx]], [tpr[optimal_idx]], color='red', s=150, zorder=5,
           label=f'Youden\'s J Optimum ({optimal_threshold:.2f})', edgecolors='black')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve - CrossEncoder Answer Routing', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.savefig(os.path.join(os.path.dirname(os.getcwd()), 'notebooks', 'plots', 'roc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Classification Report at Chosen Threshold (0.5)

In [ ]:
THRESHOLD = 0.5
y_pred = [1 if s >= THRESHOLD else 0 for s in y_scores]

print(classification_report(y_true, y_pred, target_names=['Unanswerable', 'Answerable']))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Unanswerable', 'Answerable'],
            yticklabels=['Unanswerable', 'Answerable'], ax=ax, cbar=False, annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (Threshold = {THRESHOLD})', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(os.getcwd()), 'notebooks', 'plots', 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Conclusion

### Findings
- **Answerable questions** consistently produce CrossEncoder rerank scores **> 1.0**
- **Unanswerable questions** consistently produce **negative** rerank scores (< 0)
- There is a clear separation between the two classes around the **0.0 - 0.5** range

### Chosen Threshold: **0.5**
We chose a conservative threshold of `0.5` that provides a safe margin between the distributions:
- Scores **≥ 0.5** → RAG mode (answer grounded in retrieved legal documents)
- Scores **< 0.5** → Knowledge mode (answer from the model's own training, with a disclaimer)

### Why not use Youden's J directly?
The dataset contains only a small number of unanswerable questions, making the ROC-derived
optimal point statistically noisy. The empirical threshold of 0.5 is more robust and
generalizes better to unseen queries.